# 07 — Combined frontal/posterior comparison: whole-cap SNR

Compare identity, historical frontal VMD+EOG, and calibrated frontal ridge,
crossed with posterior identity, calibrated ridge and rank1 GEVD-MWF. These
are nine fixed structural ablations, not AutoVMD or a parameter search.

Klados frontal: FP1/FP2/F3/F4/F7/F8/Fz; posterior: P3/P4/O1/O2/Pz.
Paper-list row binding remains an explicit unverified assumption. Central
C3/C4/Cz and temporal T3/T4/T5/T6 remain exactly unchanged. Full19-channel
raw/clean scoring arrays and historical frontal outputs are verified archives.

Each branch estimates correction relative to original EEG. Disjoint masks and
one subtraction form the combined output. Whole-cap SNR is measured from
that output and the clean19-channel target; regional dB are never added.
This stage is composition, not frontal-to-back message passing. No new VMD
decomposition or neural training. No LEMON or confirmation sources.

SGEYESUB lacks verified six-class Klados calibration, so no guessed paired
SGEYESUB score is produced. Native OSF has no clean target/reconstruction SNR.


## 1. Fresh exact-SHA source and environment

In [ ]:
RUN_SPEC = {'run_id': 'frontal-posterior-combined-001', 'campaign_id': 'frontal-posterior-20261017', 'kernel': 'satyapaladugu/frontal-posterior-combined-comparison', 'git_sha': 'RESOLVED_TO_PUBLISHED_SHA_AT_SUBMISSION', 'git_url': 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git', 'created_utc': '2026-10-17T09:33:17.967029+00:00', 'output': '/kaggle/working/results/frontal-posterior-20261017/frontal-posterior-combined-001'}
import os,sys,pathlib,tempfile,subprocess,json,platform
assert pathlib.Path('/kaggle/input').exists(), 'Execute numerical reporting on Kaggle'
os.environ.update(OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2')
print('BOOTSTRAP_START',RUN_SPEC['run_id'],RUN_SPEC['git_sha'],flush=True)
CHECKOUT=pathlib.Path(tempfile.mkdtemp(prefix='frontal-pretrained-source-'))
subprocess.run(['git','clone','--quiet',RUN_SPEC['git_url'],str(CHECKOUT)],check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']],check=True)
actual=subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
assert actual==RUN_SPEC['git_sha']
subprocess.run([sys.executable,'-m','pip','install','-q','pytest==8.3.5','pymatreader==1.0.0',
    'tabulate==0.9.0','torchinfo==1.8.0','PySDKit==0.5.0','condat-tv==0.0.5'],check=True,timeout=300)
sys.path.insert(0,str(CHECKOUT/'src'))
import torch,importlib.metadata
torch.set_num_threads(2)
from eog_clean.data import save_json
OUTPUT=pathlib.Path(RUN_SPEC['output']);OUTPUT.mkdir(parents=True,exist_ok=True)
save_json(OUTPUT/'run_spec.json',RUN_SPEC)
save_json(OUTPUT/'environment.json',{'git_sha':actual,'python':sys.version,'platform':platform.platform(),
    'cuda':torch.cuda.is_available(),'versions':{name:importlib.metadata.version(name) for name in
    ['numpy','scipy','pandas','torch','matplotlib']}})


## 2. Contracts, calibration and combined experiment

Run mask/identity/pooled-SNR fixtures before processing. Fit ridge0.001 and
posterior rank1 MWF at strength1 using the first2000 samples, independently
preprocessed. MWF uses bottom/top30% calibration EOG activity; these are
low/high ocular proxies, not known clean labels. No target is used to fit.
Archived historical frontal predictions and clean-input controls remain fixed.


In [ ]:
environment=os.environ.copy();environment['PYTHONPATH']=str(CHECKOUT/'src')
tests=subprocess.run([sys.executable,'-m','pytest','-q',str(CHECKOUT/'tests'),
    '--junitxml='+str(OUTPUT/'contract_tests.xml')],text=True,capture_output=True,env=environment)
(OUTPUT/'contract_tests.txt').write_text(tests.stdout+'\n'+tests.stderr)
print(tests.stdout,flush=True)
if tests.returncode:raise RuntimeError('Numerical contracts failed')
from eog_clean.regional_comparison import run,report
run('/kaggle/input',CHECKOUT,OUTPUT)
FIGURES=report(OUTPUT)


## 3. Results and qualification limits

Report frontal/posterior/whole-cap SNR and clean CC separately. Include RMSE,
temporal/spectral RRMSE, channel H/V associations, clean-input controls,
record-balanced summaries and paired record-cluster uncertainty. The eight
reused record identities are not eight verified independent participants.
Central and temporal pass-through may limit whole-cap performance; they are
not silently denoised. No final winner or deployment claim follows from SNR.


In [ ]:
import pandas as pd
from IPython.display import display,Image
table=pd.read_csv(OUTPUT/'comparison.csv')
display(table[table.group.isin(['frontal','posterior','whole_cap'])].round(6))
display(pd.read_csv(OUTPUT/'preservation_gates.csv').round(6))
display(pd.read_csv(OUTPUT/'combined_snr_uncertainty.csv').round(6))
print((OUTPUT/'method_failures.json').read_text())
print((OUTPUT/'completion.json').read_text())
for path in FIGURES:display(Image(filename=path))
save_json(OUTPUT/'execution_state.json',{'state':'complete','git_sha':actual,
    'regions_corrected':['frontal','posterior'],'central_temporal_unchanged':True,
    'regional_dB_added':False,'production_qualified':False})
